<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/final/FIN2_integration_and_workspace.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Minds, Morals and Machines

## Project 2: Integration, broadcast and indicator properties in small networks

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

The simplified integration measure of Week 11 is computed for hundreds of random boolean networks with and without feedback, and a workspace of competing modules is simulated to study ignition and global availability [1, 2].

**Required extensions.** Test systematically whether recurrence is necessary and sufficient for positive integration in the simplified measure, explain the exceptions, and compare the measure with a second one from the literature [3]. Build a small architecture that satisfies indicator properties of two theories at once and discuss what, if anything, the exercise shows about consciousness [4, 5].

## What this example implements

The notebook generates random four-node networks, computes integration as whole-system information beyond the weakest bipartition, compares acyclic and cyclic networks, and simulates ignition in a workspace of five competing modules.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Integration in random networks

In [ ]:
import itertools
rng = np.random.default_rng(SEED)

def mutual_information(states, nxt, idx):
    joint, px, py, w = {}, {}, {}, 1 / len(states)
    for s, t in zip(states, nxt):
        a, b = tuple(s[i] for i in idx), tuple(t[i] for i in idx)
        joint[(a, b)] = joint.get((a, b), 0) + w; px[a] = px.get(a, 0) + w; py[b] = py.get(b, 0) + w
    return sum(p * np.log2(p / (px[a] * py[b])) for (a, b), p in joint.items())

def integration(update, n):
    states = list(itertools.product([0, 1], repeat=n)); nxt = [update(s) for s in states]
    whole = mutual_information(states, nxt, range(n))
    parts = [whole - mutual_information(states, nxt, A) - mutual_information(states, nxt, tuple(i for i in range(n) if i not in A))
             for k in range(1, n // 2 + 1) for A in itertools.combinations(range(n), k)]
    return max(0.0, min(parts))

def random_network(n=4, k=2, feedforward=False):
    inputs, tables = [], []
    for i in range(n):
        pool = list(range(i)) if feedforward else list(range(n))
        if not pool:
            inputs.append([i]); tables.append({(0,): 0, (1,): 1}); continue
        src = [int(x) for x in rng.choice(pool, size=min(k, len(pool)), replace=False)]
        inputs.append(src); tables.append({bits: int(rng.integers(2)) for bits in itertools.product([0, 1], repeat=len(src))})
    return (lambda s: tuple(tables[i][tuple(s[j] for j in inputs[i])] for i in range(n))), inputs

def has_cycle(inputs):
    n = len(inputs); succ = {i: [j for j in range(n) if i in inputs[j] and j != i] for i in range(n)}
    def reach(a, b, seen=()):
        return any(c == b or (c not in seen and reach(c, b, seen + (c,))) for c in succ[a])
    return any(reach(i, i) for i in range(n))

res = []
for ff in (True, False):
    for _ in range(150):
        up, inp = random_network(feedforward=ff)
        res.append(dict(feedforward=ff, cycle=has_cycle(inp), integration=integration(up, 4)))
import pandas as pd
df = pd.DataFrame(res)
print(df.groupby(["feedforward", "cycle"])["integration"].describe()[["count", "mean", "max"]].round(3))

## 2. Ignition in a competitive workspace

In [ ]:
def workspace(strengths, w_self=6.0, w_inh=4.0, theta=3.0, temp=0.3, steps=300, dt=0.05):
    a = np.zeros(len(strengths))
    for _ in range(steps):
        a += dt * (-a + 1 / (1 + np.exp(-(w_self * a - w_inh * (a.sum() - a) + strengths - theta) / temp)))
    return a

levels = np.linspace(0, 3, 31)
ignite, winners = [], []
for s in levels:
    trials = [workspace(np.r_[s, rng.uniform(0, 1.5, 4)]) for _ in range(20)]
    ignite.append(np.mean([(t > 0.5).sum() == 1 for t in trials])); winners.append(np.mean([np.argmax(t) == 0 and t[0] > 0.5 for t in trials]))
plt.plot(levels, ignite, label="exactly one module ignites"); plt.plot(levels, winners, label="the target module wins the workspace")
plt.xlabel("strength of the target input"); plt.ylabel("share of trials"); plt.legend(); plt.show()

## Report and submission

Both tracks follow the course report template. A research report states its thesis in the introduction, reconstructs the relevant arguments as explicit premises and conclusions, and evaluates them. An application report explains what the simulation or experiment can and cannot show about the philosophical question, presents its results with figures and discusses its limitations. All sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Tononi, G. (2004). An information integration theory of consciousness. *BMC Neuroscience*, 5, 42. <https://doi.org/10.1186/1471-2202-5-42>

[2] Dehaene, S., & Naccache, L. (2001). Towards a cognitive neuroscience of consciousness: Basic evidence and a workspace framework. *Cognition*, 79(1-2), 1-37. <https://doi.org/10.1016/S0010-0277(00)00123-2>

[3] Albantakis, L., Barbosa, L., Findlay, G., et al. (2023). Integrated information theory (IIT) 4.0: Formulating the properties of phenomenal existence in physical terms. *PLOS Computational Biology*, 19(10), e1011465. <https://doi.org/10.1371/journal.pcbi.1011465>

[4] Butlin, P., Long, R., Elmoznino, E., Bengio, Y., Birch, J., Constant, A., Deane, G., Fleming, S. M., Frith, C., Ji, X., Kanai, R., Klein, C., Lindsay, G., Michel, M., Mudrik, L., Peters, M. A. K., Schwitzgebel, E., Simon, J., & VanRullen, R. (2023). Consciousness in artificial intelligence: Insights from the science of consciousness. arXiv preprint arXiv:2308.08708. <https://arxiv.org/abs/2308.08708>

[5] Seth, A. K. (2025). Conscious artificial intelligence and biological naturalism. *Behavioral and Brain Sciences*. Published online 21 April 2025. <https://doi.org/10.1017/S0140525X25000032>